# 02 - Statistical Baseline (safety net)

Flattens **Bohra 2018 + HASOC 2021 + HASOC 2022** into one frame `[text, label, source, script]`, filters to **romanised** rows, and runs a **TF-IDF + Logistic Regression** baseline.

Outputs two things:
1. **within-Bohra macro-F1** - the safety-net number Hunter asked for.
2. **cross-dataset macro-F1** (train Bohra, test HASOC) - the generalisation drop the dissertation is about.

**Before running:** put the `hinglish_hate/` package folder next to this notebook (upload both to Colab, or keep them in the same Drive folder). Then set `DATA_ROOT` in the config cell to the folder that holds `hate_speech.tsv`, `final.csv`, and the HASOC-2021 thread folders.

### 1. Mount Drive (skip if running locally)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not on Colab / already mounted:', e)

### 2. Make the package importable and import it

In [ ]:
import sys, os
# add this notebook's folder (and CWD) so `import hinglish_hate` works
for p in {os.getcwd(), os.path.dirname(os.path.abspath('__file__'))}:
    if p and p not in sys.path:
        sys.path.insert(0, p)

from hinglish_hate import (
    load_bohra, load_hasoc2021, load_hasoc2022,
    build_corpus, filter_romanised,
    evaluate_within, evaluate_cross, summarise,
)
import pandas as pd
print('package imported OK')

### 3. Locate the data files

The HASOC folder names are long and machine-generated, so instead of hardcoding them we search under `DATA_ROOT` by filename. Point `DATA_ROOT` at wherever the dataset folder lives once Drive is mounted.

In [ ]:
from pathlib import Path

# EDIT THIS: the folder that contains hate_speech.tsv, final.csv, HASOC-2021 threads.
# Example after mounting a shared folder added to My Drive:
# DATA_ROOT = Path('/content/drive/MyDrive/Hinglish_Datasets')
DATA_ROOT = Path('.')

def find_one(root, name):
    hits = list(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {root}')
    return hits[0]

bohra_path = find_one(DATA_ROOT, 'hate_speech.tsv')
h22_path   = find_one(DATA_ROOT, 'final.csv')
# HASOC-2021: any ancestor that contains data.json/labels.json thread folders.
# Point at the 2021 split root; loader walks the tree. Adjust name if needed.
h21_labels = list(Path(DATA_ROOT).rglob('labels.json'))
h21_root   = h21_labels[0].parents[2] if h21_labels else None

print('Bohra    :', bohra_path)
print('HASOC22  :', h22_path)
print('HASOC21  :', h21_root, f'({len(h21_labels)} thread label files)')

### 4. Load and profile each source

Mirrors the label/script breakdown from `01_data_exploration`.

In [ ]:
bohra = load_bohra(bohra_path)
h22   = load_hasoc2022(h22_path)
h21   = load_hasoc2021(h21_root) if h21_root else None

for name, df in [('bohra2018', bohra), ('hasoc2021', h21), ('hasoc2022', h22)]:
    if df is None:
        print(f'{name:11s} - skipped'); continue
    print(f"{name:11s} rows={len(df):5d}  hate%={df['label'].mean():.2f}")
    print('   scripts:', dict(df['script'].value_counts()))

### 5. Unify + filter to romanised

`include_mixed=True` keeps code-switched posts that carry some Devanagari; set it to `False` for Latin-only. Bohra is already all romanised; the HASOC sources lose their pure-Devanagari rows here.

In [ ]:
frames = [f for f in [bohra, h21, h22] if f is not None]
corpus = build_corpus(frames)
roman  = filter_romanised(corpus, include_mixed=True)

print('full corpus  :', len(corpus))
print('romanised    :', len(roman))
print()
print(pd.crosstab(roman['source'], roman['label']).rename(columns={0:'not',1:'hate'}))

## 6. Safety-net baseline (within Bohra)

**This is the number to report first.** Stratified 80/20 split on the romanised Bohra set, TF-IDF + Logistic Regression, macro-F1.

In [ ]:
bohra_r = filter_romanised(bohra)  # Latin-only
within = evaluate_within(bohra_r)
print(summarise('within Bohra', within))
print()
print(within['report'])

## 7. Cross-dataset generalisation

Train on Bohra, test on each HASOC set (romanised). The gap between the within-Bohra F1 above and these is the generalisation drop - the core finding the project is built to measure.

In [ ]:
h22_r = filter_romanised(h22, include_mixed=True)
print(summarise('Bohra -> HASOC22', evaluate_cross(bohra_r, h22_r)))

if h21 is not None:
    h21_r = filter_romanised(h21, include_mixed=True)
    if len(h21_r):
        print(summarise('Bohra -> HASOC21', evaluate_cross(bohra_r, h21_r)))

### Notes
- `class_weight='balanced'` handles the hate/not imbalance; macro-F1 is the headline so both classes count equally.
- Everything is seeded (`random_state=42`) so numbers are reproducible.
- This baseline is the floor every later model (XLM-R, IndicBERT, MuRIL, the LoRA LLM) has to beat. Keep the exact `roman`/`bohra_r` frames so the test sets are identical across models.